# MedMNIST v2 — All Remaining Work (One Notebook)

This single notebook produces everything still missing from the project:

1. **PathMNIST Phase-1 analysis** — class distribution table + bar chart + montage (secondary dataset characterization)
2. **Person C** — ResNet-18 on PathMNIST (28×28) reproduction + per-class analysis
3. **Person B** — ResNet-50 on DermaMNIST (--resize 28→224) reproduction + per-class analysis

Order is deliberate: the cheap Phase-1 runs first, then the *fast* Person C (~1 h), then the *slow* Person B (~3–5 h) last — so if the long ResNet-50 run fails you still keep the earlier results.

**Runtime (all sequential in one session):** ~4–6 h total — fits comfortably in one 12 h Kaggle session.

## Before you run — Kaggle settings
- **Accelerator:** `GPU T4 x2`  (NOT P100 — Kaggle's PyTorch dropped Pascal/sm_60 support)
- **Internet:** `ON`  (datasets auto-download via the `medmnist` package)
- **No datasets need to be attached.** Everything downloads itself. Just use **Save & Run All (Commit)** for an unattended run.

## Outputs (download from the notebook's Output tab when done)
- `phase1/pathmnist_class_distribution.csv`, `phase1/pathmnist_class_distribution.png`, `phase1/pathmnist_montage.png`, `phase1/pathmnist.csv`
- `perclass_pathmnist_resnet18.csv`, `confusion_matrix_resnet18_pathmnist.png`, `perclass_metrics_resnet18_pathmnist.png`, `reproduction_summary_personC.csv`
- `perclass_dermamnist_resnet50.csv`, `confusion_matrix_resnet50_dermamnist.png`, `perclass_metrics_resnet50_dermamnist.png`, `reproduction_summary_personB.csv`


## Setup (shared) — GPU check, install, clone experiments repo

### Cell 1 — GPU Check

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU — Settings -> Accelerator -> GPU T4 x2")

name   = torch.cuda.get_device_name(0)
cap    = torch.cuda.get_device_capability(0)   # (major, minor)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU            : {name}")
print(f"Compute cap.   : sm_{cap[0]}{cap[1]}")
print(f"VRAM           : {mem_gb:.1f} GB")

# Kaggle's current PyTorch build DROPPED Pascal (sm_60), so the P100 crashes at
# the first forward pass. The T4 (sm_75) works. Fail fast with a clear message.
if cap[0] < 7:
    raise RuntimeError(
        f"{name} (sm_{cap[0]}{cap[1]}) is too old for Kaggle's current PyTorch. "
        "Switch Accelerator to 'GPU T4 x2' in notebook settings and rerun."
    )
print("GPU is compatible with the current PyTorch build.")


### Cell 2 — Install Missing Packages

In [ ]:
# Kaggle pre-installs torch, torchvision, numpy, pandas, scikit-learn, Pillow.
# We only add these three:
!pip install -q medmnist tensorboardX fire
print("Packages installed.")


### Cell 3 — Clone Experiments Repo

In [ ]:
import os

os.chdir('/kaggle/working')
if not os.path.isdir('/kaggle/working/experiments'):
    !git clone https://github.com/MedMNIST/experiments
os.chdir('/kaggle/working/experiments/MedMNIST2D')
os.makedirs('/kaggle/working/output', exist_ok=True)
os.makedirs('/kaggle/working/phase1', exist_ok=True)
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))


## Part 1 — PathMNIST Phase-1 Analysis
Characterize the secondary dataset the same way DermaMNIST was characterized in Phase 1:
class-distribution table + bar chart + sample montage + per-image manifest CSV.
PathMNIST is 9 tissue classes and much more balanced than DermaMNIST — a useful control
for the bias narrative. No training here; runs in a couple of minutes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from medmnist import PathMNIST

PATH_CLASSES = [
    'Adipose', 'Background', 'Debris', 'Lymphocytes', 'Mucus',
    'Smooth muscle', 'Normal colon mucosa', 'Cancer-associated stroma',
    'Colorectal adenocarcinoma epithelium',
]
N_PATH = 9

# Load each split (labels only needed for distribution + manifest)
path_splits = {}
for sp in ['train', 'val', 'test']:
    ds = PathMNIST(split=sp, download=True)
    path_splits[sp] = ds.labels.flatten().astype(int)
    print(f"{sp:<5} : {len(path_splits[sp]):>6} images")

# ---- Class-distribution table ----
dist = pd.DataFrame({
    'Class': PATH_CLASSES,
    'Train': np.bincount(path_splits['train'], minlength=N_PATH),
    'Val':   np.bincount(path_splits['val'],   minlength=N_PATH),
    'Test':  np.bincount(path_splits['test'],  minlength=N_PATH),
})
dist['Total']    = dist[['Train', 'Val', 'Test']].sum(axis=1)
dist['Train %']  = (dist['Train'] / dist['Train'].sum() * 100).round(2)
print("\n=== PathMNIST class distribution ===")
print(dist.to_string(index=False))
dist.to_csv('/kaggle/working/phase1/pathmnist_class_distribution.csv', index=False)

# ---- Per-image manifest CSV (mirrors dermamnist.csv format) ----
man = []
for sp in ['train', 'val', 'test']:
    for idx, lab in enumerate(path_splits[sp]):
        man.append([sp.upper(), f'{sp}{idx}_{lab}.png', lab])
pd.DataFrame(man).to_csv('/kaggle/working/phase1/pathmnist.csv', index=False, header=False)
print("\nSaved: phase1/pathmnist_class_distribution.csv  and  phase1/pathmnist.csv")


In [ ]:
# ---- Grouped bar chart of class distribution by split ----
x = np.arange(N_PATH)
w = 0.27
fig, ax = plt.subplots(figsize=(13, 5))
for k, sp in enumerate(['train', 'val', 'test']):
    counts = np.bincount(path_splits[sp], minlength=N_PATH)
    ax.bar(x + (k - 1) * w, counts, w, label=sp, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(PATH_CLASSES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Number of images')
ax.set_title('PathMNIST — Class Distribution by Split')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/phase1/pathmnist_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: phase1/pathmnist_class_distribution.png")


In [ ]:
# ---- Sample montage (built into medmnist datasets) ----
montage = PathMNIST(split='train', download=True).montage(length=20)
montage.save('/kaggle/working/phase1/pathmnist_montage.png')
plt.figure(figsize=(8, 8))
plt.imshow(montage)
plt.axis('off')
plt.title('PathMNIST — Sample Montage (train)')
plt.show()
print("Saved: phase1/pathmnist_montage.png")


## Part 2 — Person C: ResNet-18 on PathMNIST (28×28)

**Target metrics (paper Table 1, ResNet-18 @ 28):** AUC ~0.983, Acc ~0.907
**Expected runtime:** ~40 min – 1.5 h on T4 (large dataset, tiny images, light model).

### Cell — Training (Person C)

In [ ]:
import time, os

os.chdir('/kaggle/working/experiments/MedMNIST2D')
torch.cuda.reset_peak_memory_stats()
t_start = time.time()

!python train_and_eval_pytorch.py \
    --data_flag pathmnist \
    --output_root /kaggle/working/output \
    --num_epochs 100 \
    --gpu_ids 0 \
    --batch_size 128 \
    --model_flag resnet18 \
    --download \
    --run run1

t_total = time.time() - t_start
print(f"\nPerson C training time : {t_total/3600:.2f} h  ({t_total/60:.0f} min)")
print(f"Approx time/epoch      : {t_total/100/60:.1f} min")

peak = torch.cuda.max_memory_allocated(0) / 1e9
print(f"Peak GPU memory        : {peak:.2f} GB")


### Cell — Read Aggregate Log (Person C)

In [ ]:
import glob

log_files = glob.glob('/kaggle/working/output/pathmnist/**/*.txt', recursive=True)
print(f"Found {len(log_files)} log file(s)\n")
for f in log_files:
    print(f"=== {f} ===")
    with open(f) as fp:
        print(fp.read())
print("\n--- Paper targets ---  AUC ~0.983   Acc ~0.907")


### Cell — Per-Class Analysis (Person C)

In [ ]:
import PIL
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet18
from medmnist import PathMNIST
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.preprocessing import label_binarize

CLASS_NAMES = PATH_CLASSES
N_CLASSES   = 9
DEVICE      = torch.device('cuda:0')

model_files = glob.glob('/kaggle/working/output/pathmnist/**/best_model.pth', recursive=True)
assert model_files, "best_model.pth not found — check that Person C training completed"
model_path = sorted(model_files)[-1]
print(f"Loading: {model_path}")

model = resnet18(weights=None, num_classes=N_CLASSES).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE)['net'])
model.eval()

# No --resize: native 28x28
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
])
test_ds     = PathMNIST(split='test', transform=transform, download=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

all_scores, all_targets = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        scores = torch.softmax(model(inputs.to(DEVICE)), dim=1)
        all_scores.append(scores.cpu().numpy())
        all_targets.append(targets.numpy().flatten())

y_score = np.vstack(all_scores)
y_true  = np.concatenate(all_targets)
y_pred  = y_score.argmax(axis=1)

y_bin   = label_binarize(y_true, classes=list(range(N_CLASSES)))
per_auc = [roc_auc_score(y_bin[:, i], y_score[:, i]) for i in range(N_CLASSES)]
cm      = confusion_matrix(y_true, y_pred)
per_acc = cm.diagonal() / cm.sum(axis=1)
per_f1   = f1_score(y_true, y_pred, average=None, zero_division=0)
per_prec = precision_score(y_true, y_pred, average=None, zero_division=0)
per_rec  = recall_score(y_true, y_pred, average=None, zero_division=0)
counts   = np.bincount(y_true, minlength=N_CLASSES)

df = pd.DataFrame({
    'Class': CLASS_NAMES, 'N (test)': counts,
    'AUC': np.round(per_auc, 4), 'Accuracy': np.round(per_acc, 4),
    'Precision': np.round(per_prec, 4), 'Recall': np.round(per_rec, 4),
    'F1': np.round(per_f1, 4),
})
print("\n=== Per-Class — ResNet-18, PathMNIST (Test) ===")
print(df.to_string(index=False))
print(f"\n  Macro AUC : {np.mean(per_auc):.4f}   (paper ~0.983)")
print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}   (paper ~0.907)")
print(f"  Macro F1  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
df.to_csv('/kaggle/working/perclass_pathmnist_resnet18.csv', index=False)
print("\nSaved: perclass_pathmnist_resnet18.csv")

# keep C's aggregates for the final summary
C_auc, C_acc = float(np.mean(per_auc)), float(accuracy_score(y_true, y_pred))


### Cell — Confusion Matrix + Bar Charts (Person C)

In [ ]:
import matplotlib.pyplot as plt

short = ['Adipose', 'Background', 'Debris', 'Lympho-\ncytes', 'Mucus',
         'Smooth\nmuscle', 'Normal\nmucosa', 'CA\nstroma', 'Adenoca.\nepith.']

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(short, fontsize=7, rotation=45, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(short, fontsize=7)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('ResNet-18 — PathMNIST — Confusion Matrix (Test)')
thresh = cm.max() / 2
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6,
                color='white' if cm[i, j] > thresh else 'black')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_resnet18_pathmnist.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].bar(range(N_CLASSES), per_auc, edgecolor='white')
axes[0].axhline(np.mean(per_auc), color='black', ls='--', lw=1, label=f'Macro AUC = {np.mean(per_auc):.3f}')
axes[0].axhline(0.983, color='red', ls=':', lw=1, label='Paper (0.983)')
axes[0].set_xticks(range(N_CLASSES)); axes[0].set_xticklabels(short, fontsize=7, rotation=45, ha='right')
axes[0].set_ylim(0.5, 1.0); axes[0].set_ylabel('AUC'); axes[0].set_title('Per-Class AUC'); axes[0].legend(fontsize=8)
axes[1].bar(range(N_CLASSES), per_acc, edgecolor='white')
axes[1].axhline(C_acc, color='black', ls='--', lw=1, label=f'Overall Acc = {C_acc:.3f}')
axes[1].axhline(0.907, color='red', ls=':', lw=1, label='Paper (0.907)')
axes[1].set_xticks(range(N_CLASSES)); axes[1].set_xticklabels(short, fontsize=7, rotation=45, ha='right')
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Per-Class Accuracy'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('/kaggle/working/perclass_metrics_resnet18_pathmnist.png', dpi=150, bbox_inches='tight')
plt.show()

summaryC = pd.DataFrame({
    'Config': ['ResNet-18, PathMNIST, 28x28'],
    'AUC (paper)': [0.983], 'AUC (ours)': [round(C_auc, 4)], 'AUC diff': [round(C_auc - 0.983, 4)],
    'Acc (paper)': [0.907], 'Acc (ours)': [round(C_acc, 4)], 'Acc diff': [round(C_acc - 0.907, 4)],
})
print(summaryC.to_string(index=False))
summaryC.to_csv('/kaggle/working/reproduction_summary_personC.csv', index=False)
print("Saved Person C outputs.")


## Part 3 — Person B: ResNet-50 on DermaMNIST (--resize 28→224)

**Target metrics (paper Table 1, ResNet-50 @ 224):** AUC ~0.912, Acc ~0.731
**Expected runtime:** ~3–5 h on T4 (heaviest job — this is the long pole). Runs last on purpose.

### Cell — Training (Person B)

In [ ]:
import time, os

os.chdir('/kaggle/working/experiments/MedMNIST2D')
torch.cuda.reset_peak_memory_stats()
t_start = time.time()

!python train_and_eval_pytorch.py \
    --data_flag dermamnist \
    --output_root /kaggle/working/output \
    --num_epochs 100 \
    --gpu_ids 0 \
    --batch_size 128 \
    --resize \
    --model_flag resnet50 \
    --download \
    --run run1

t_total = time.time() - t_start
print(f"\nPerson B training time : {t_total/3600:.2f} h  ({t_total/60:.0f} min)")
print(f"Approx time/epoch      : {t_total/100/60:.1f} min")
peak = torch.cuda.max_memory_allocated(0) / 1e9
print(f"Peak GPU memory        : {peak:.2f} GB")


### Cell — Read Aggregate Log (Person B)

In [ ]:
import glob

log_files = glob.glob('/kaggle/working/output/dermamnist/**/*.txt', recursive=True)
print(f"Found {len(log_files)} log file(s)\n")
for f in log_files:
    print(f"=== {f} ===")
    with open(f) as fp:
        print(fp.read())
print("\n--- Paper targets ---  AUC ~0.912   Acc ~0.731")


### Cell — Per-Class Analysis (Person B)

In [ ]:
import PIL
from torchvision.models import resnet50
from medmnist import DermaMNIST

DERMA_CLASSES = [
    'Actinic keratoses', 'Basal cell carcinoma', 'Benign keratosis',
    'Dermatofibroma', 'Melanoma', 'Melanocytic nevi', 'Vascular lesions',
]
N_CLASSES = 7
DEVICE    = torch.device('cuda:0')

model_files = glob.glob('/kaggle/working/output/dermamnist/**/best_model.pth', recursive=True)
assert model_files, "best_model.pth not found — check that Person B training completed"
model_path = sorted(model_files)[-1]
print(f"Loading: {model_path}")

model = resnet50(weights=None, num_classes=N_CLASSES).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE)['net'])
model.eval()

# --resize training => eval at 224x224
transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=PIL.Image.NEAREST),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
])
test_ds     = DermaMNIST(split='test', transform=transform, download=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

all_scores, all_targets = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        scores = torch.softmax(model(inputs.to(DEVICE)), dim=1)
        all_scores.append(scores.cpu().numpy())
        all_targets.append(targets.numpy().flatten())

y_score = np.vstack(all_scores)
y_true  = np.concatenate(all_targets)
y_pred  = y_score.argmax(axis=1)

y_bin   = label_binarize(y_true, classes=list(range(N_CLASSES)))
per_auc = [roc_auc_score(y_bin[:, i], y_score[:, i]) for i in range(N_CLASSES)]
cm      = confusion_matrix(y_true, y_pred)
per_acc = cm.diagonal() / cm.sum(axis=1)
per_f1   = f1_score(y_true, y_pred, average=None, zero_division=0)
per_prec = precision_score(y_true, y_pred, average=None, zero_division=0)
per_rec  = recall_score(y_true, y_pred, average=None, zero_division=0)
counts   = np.bincount(y_true, minlength=N_CLASSES)

df = pd.DataFrame({
    'Class': DERMA_CLASSES, 'N (test)': counts,
    'AUC': np.round(per_auc, 4), 'Accuracy': np.round(per_acc, 4),
    'Precision': np.round(per_prec, 4), 'Recall': np.round(per_rec, 4),
    'F1': np.round(per_f1, 4),
})
print("\n=== Per-Class — ResNet-50, DermaMNIST (Test) ===")
print(df.to_string(index=False))
print(f"\n  Macro AUC : {np.mean(per_auc):.4f}   (paper ~0.912)")
print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}   (paper ~0.731)")
print(f"  Macro F1  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
df.to_csv('/kaggle/working/perclass_dermamnist_resnet50.csv', index=False)
print("\nSaved: perclass_dermamnist_resnet50.csv")

B_auc, B_acc = float(np.mean(per_auc)), float(accuracy_score(y_true, y_pred))


### Cell — Confusion Matrix + Bar Charts (Person B)

In [ ]:
short = ['Actinic\nkerat.', 'Basal cell\ncarc.', 'Benign\nkerat.',
         'Dermato-\nfibroma', 'Melanoma', 'Melanocytic\nnevi', 'Vascular\nlesions']

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(short, fontsize=7, rotation=45, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(short, fontsize=7)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('ResNet-50 — DermaMNIST — Confusion Matrix (Test)')
thresh = cm.max() / 2
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                color='white' if cm[i, j] > thresh else 'black')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix_resnet50_dermamnist.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].bar(range(N_CLASSES), per_auc, edgecolor='white')
axes[0].axhline(np.mean(per_auc), color='black', ls='--', lw=1, label=f'Macro AUC = {np.mean(per_auc):.3f}')
axes[0].axhline(0.912, color='red', ls=':', lw=1, label='Paper (0.912)')
axes[0].set_xticks(range(N_CLASSES)); axes[0].set_xticklabels(short, fontsize=7, rotation=45, ha='right')
axes[0].set_ylim(0.5, 1.0); axes[0].set_ylabel('AUC'); axes[0].set_title('Per-Class AUC'); axes[0].legend(fontsize=8)
axes[1].bar(range(N_CLASSES), per_acc, edgecolor='white')
axes[1].axhline(B_acc, color='black', ls='--', lw=1, label=f'Overall Acc = {B_acc:.3f}')
axes[1].axhline(0.731, color='red', ls=':', lw=1, label='Paper (0.731)')
axes[1].set_xticks(range(N_CLASSES)); axes[1].set_xticklabels(short, fontsize=7, rotation=45, ha='right')
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Per-Class Accuracy'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('/kaggle/working/perclass_metrics_resnet50_dermamnist.png', dpi=150, bbox_inches='tight')
plt.show()

summaryB = pd.DataFrame({
    'Config': ['ResNet-50, DermaMNIST, --resize'],
    'AUC (paper)': [0.912], 'AUC (ours)': [round(B_auc, 4)], 'AUC diff': [round(B_auc - 0.912, 4)],
    'Acc (paper)': [0.731], 'Acc (ours)': [round(B_acc, 4)], 'Acc diff': [round(B_acc - 0.731, 4)],
})
print(summaryB.to_string(index=False))
summaryB.to_csv('/kaggle/working/reproduction_summary_personB.csv', index=False)
print("Saved Person B outputs.")


## Done — file manifest to download

In [ ]:
import glob, os
print("=== Files produced (download from the notebook Output tab) ===\n")
patterns = ['/kaggle/working/phase1/*', '/kaggle/working/*.csv', '/kaggle/working/*.png']
for pat in patterns:
    for f in sorted(glob.glob(pat)):
        if os.path.isfile(f):
            print(f"  {f:<70} {os.path.getsize(f)/1024:>8.1f} KB")
